# Fieldnotes · Corpus Studio

## Introduction

Fieldnotes helps you explore a language through recorded examples. Start with an annotated recording, read its transcription alongside a translation, find recurring words, and collect those words in a lexicon with the passages where they occur.

This notebook is a hands-on walkthrough of the Python tools behind the workspace. The sample uses original-language transcriptions and Portuguese translations. It covers text search, word counts, timestamped tables, annotation metadata, and a persistent word bank.

**Reading online?** The published page shows Markdown, code, and saved outputs. To run or change the examples, download the notebook and open it with the rest of this repository in Jupyter or VS Code.

## How to use this notebook

### 1. Open the project locally

Clone or download the repository. Open `test.ipynb` from the repository root and select a Python environment with `pandas`, `nbformat`, and `nbconvert` installed. For the browser-based notebook editor, install `jupyterlab` as well.

```sh
python3 -m venv .venv
source .venv/bin/activate
python -m pip install pandas jupyterlab nbformat nbconvert
jupyter lab test.ipynb
```

In VS Code, choose the `.venv` interpreter using **Select Kernel**. Run cells from top to bottom with **Shift+Enter**. Later cells reuse variables such as `df` from earlier cells.

### 2. Understand the sample files

The sample record lives in `data/eaf_sample_1/`:

| File | Purpose |
| --- | --- |
| `eaf_example.eaf` | Annotation text, tiers, references, and timestamps |
| `pfsx_example.pfsx` | ELAN display preferences |
| `des-20150619-jbrm-aranha-paidospeixes.mp4` | Recording supplied with this record |
| `The hunter and the dwarf.xml` | Additional XML document; it is not automatically aligned by these examples |

The examples below read the EAF file. Its `JBMassa-des` tier contains the transcription, `Traducao` contains Portuguese translations, and `Morfemas` / `Morf Glossa` contain morpheme and gloss annotations. Some annotations are empty.

### 3. Follow the examples

Work through the numbered sections below:

1. **Find text:** replace `ĩyẽ` with a word or phrase you want to locate.
2. **Count words:** see each distinct whitespace-separated word and its frequency.
3. **Join translations:** read original text and Portuguese translations in the same row.
4. **Explore segments:** inspect timestamps, speaker information, and child annotations.
5. **Inspect attributes:** view XML attribute names as DataFrame columns.
6. **Search by meaning or sound location:** search Portuguese or original-language text and get annotated recording intervals.
7. **Build a lexicon:** save a word with its examples and add your own meaning and notes.

### 4. Interpret the results

Times ending in `_ms` are milliseconds; divide by 1,000 for seconds. Blank translations mean no translation was recorded for that segment. `None` or `NaT` means the file does not provide an explicit time.

A Portuguese search finds matching **sentences**, not a verified word-to-word translation. Audio search finds **annotated intervals**; it does not listen to the recording or recognize untranscribed speech. Original-language accents remain meaningful, even when equivalent Unicode spellings are normalized.

### 5. Save your work

Most cells only read the corpus. The lexicon example writes to `data/lexicon.json` when you run `lexicon.add(...)`. It retains contextual examples and merges duplicates when you add the same word again. Supply a meaning only when you have established it.

Save the notebook after running cells to keep their outputs. Push the notebook to `main` to update the GitHub Pages walkthrough with those saved results.

## Prefer the visual workspace?

Run `python app.py` from the project root and open `http://127.0.0.1:8765`.

- **Volume:** connect a local folder or configured cloud location. Arrange record assets inside top-level folders named by `record_id`.
- **Corpus explorer:** select a record, read the annotations, and choose an available translation tier. Recording playback depends on the record's supported media.
- **Labeling workspace:** select an original-language word and review possible Portuguese meanings against sentence evidence.
- **My lexicon:** review collected words, their meanings, and saved examples.

The local app runs separately from this published notebook page. See the repository README for storage dependencies and setup.


## 1. Find a word or phrase

Search EAF annotation text by word or phrase. Matches are literal substrings, with Unicode accents normalized.

In [ ]:
from linguisticdocs.eaf_search import search_eaf

matches = search_eaf("data/eaf_sample_1/eaf_example.eaf", "ĩyẽ")
matches

In [ ]:
# Search a phrase within the transcription tier only.
search_eaf("data/eaf_sample_1/eaf_example.eaf", "ĩyẽ mʉ̃ãrẽ", tier="JBMassa-des")

## 2. Count unique words

Count whitespace-separated words in the transcription tier. Accents, punctuation, and case are preserved; equivalent Unicode spellings are combined. Total words counts occurrences; unique words counts distinct spellings.

In [ ]:
from linguisticdocs.eaf_search import count_eaf_words

word_counts = count_eaf_words("data/eaf_sample_1/eaf_example.eaf", tier="JBMassa-des")
print("Total words:", sum(word_counts.values()))
print("Unique words:", len(word_counts))
word_counts.most_common()  # (word, count), most frequent first

In [ ]:
# For multiple EAF files with the same transcription tier:
# from pathlib import Path
# word_counts = count_eaf_words(sorted(Path("data").rglob("*.eaf")), tier="JBMassa-des")

## 3. Put translations beside the original text

Display the transcription and its Portuguese translation as columns. Rows are joined by annotation reference; untranslated segments remain in the table with a blank translation.

In [ ]:
import importlib
from linguisticdocs import eaf_search
import pandas as pd

importlib.reload(eaf_search)  # Pick up changes if already imported.
rows = eaf_search.eaf_translation_rows("data/eaf_sample_1/eaf_example.eaf")
df = pd.DataFrame(rows, columns=["annotation_id", "start_ms", "end_ms", "text", "translation"])
df.head(10)

## 4. Explore timestamped segments

Segment table: one row per transcription interval, with translation, morphemes, glosses, and metadata. associated_annotations retains child IDs, timing, parent links, and tier metadata. Unknown child times remain None.

In [ ]:
from linguisticdocs.eaf_dataframe import eaf_dataframe

df = eaf_dataframe("data/eaf_sample_1/eaf_example.eaf")
display(df[["timestamp", "end_timestamp", "transcript", "translation", "morphemes", "glosses", "participant"]])

In [ ]:
# Inspect every annotation associated with the second segment.
df.loc[1, "associated_annotations"]

## 5. Inspect annotation attributes

Attribute table: one row per annotation, original XML attribute names as columns, and start timestamp as the index. Repeated timestamps represent separate annotations. NaT indicates unknown timing. ANNOTATION_VALUE is the annotation text, included alongside the attributes.

In [ ]:
from linguisticdocs.eaf_dataframe import eaf_attribute_dataframe

attributes_df = eaf_attribute_dataframe("data/eaf_sample_1/eaf_example.eaf")
display(attributes_df)

## 6. Search translations and recording intervals

Search Portuguese translations or original-language transcripts. Whole-word matching is the default; use whole_word=False for substrings. Audio search returns annotated source-language segment intervals, with matching morpheme intervals where available. It does not recognize untranscribed speech.

In [ ]:
from linguisticdocs.eaf_dataframe import eaf_dataframe
from linguisticdocs.corpus_search import search_translation, search_transcript, search_audio

df = eaf_dataframe("data/eaf_sample_1/eaf_example.eaf")
columns = ["timestamp", "end_timestamp", "transcript", "translation"]

# Portuguese: estoria also matches estória.
display(search_translation(df, "estoria")[columns])

In [ ]:
# Original-language word or phrase (accents preserved).
display(search_transcript(df, "ĩyẽ")[columns])

In [ ]:
# Audio intervals for an original-language word.
audio_matches = search_audio(df, "ĩyẽ")
display(audio_matches[columns + ["start_seconds", "end_seconds", "timing_scope", "matching_child_intervals"]])

# Locate source audio segments using their Portuguese translation.
display(search_audio(df, "estoria", language="translation")[columns + ["start_seconds", "end_seconds"]])

## 7. Build your word bank

Persistent lexicon / word bank. Running add saves to data/lexicon.json. Examples include source file, transcript, sentence translation, segment timestamps and available child intervals. Enter lexical meanings yourself; sentence translations are not word definitions. Re-adding a word merges examples without duplicates.

In [ ]:
from lexicon import Lexicon
from linguisticdocs.eaf_dataframe import eaf_dataframe

df = eaf_dataframe("data/eaf_sample_1/eaf_example.eaf")
lexicon = Lexicon("data/lexicon.json")

# Save a word with its automatically identified corpus examples.
entry = lexicon.add(
    "ĩyẽ", df,
    source_file="data/eaf_sample_1/eaf_example.eaf",
    notes="Meaning to be reviewed",
)
display(lexicon.to_dataframe())
display(lexicon.examples("ĩyẽ"))

In [ ]:
# Update a meaning when you have established it:
# lexicon.add("ĩyẽ", df, source_file="data/eaf_sample_1/eaf_example.eaf",
#             meaning="Enter the verified meaning here", part_of_speech="...")

# Optionally add only selected matching examples:
# lexicon.add("ĩyẽ", df, source_file="data/eaf_sample_1/eaf_example.eaf", example_ids=["a1"])

# Reopen the same file in a later notebook session:
# lexicon = Lexicon("data/lexicon.json")
# display(lexicon.to_dataframe())